# Calculation — Hamiltonian Poincare section, implicit BM4, 16 parallel processes

**48 radial particles · 5,000 forcing cycles · 20 steps/cycle · 16 processes · no reference.**
This notebook computes and saves the numerical data. Open `visualizacion.ipynb`
afterwards to produce plots and the cycle viewer without repeating the calculation.

The complete folder is portable: the field and original BM4 implementation are
in `assets/gc2d_snapshot.zip`. No local repository or original HDF5 file is needed.
Use Python 3.11 or 3.12 with `python -m pip install -r requirements.txt`, then
run this notebook from its folder. For EC2 or unattended execution:
`python run.py calculate --run-id aws_48p_5000c_20s_16proc_spot_20260920`.

All result files are saved under `resultados/<RUN_ID>/` in this folder. Each
run ID is unique; an existing calculation is never overwritten. The CLI runner
also saves its log, status and an executed copy of this notebook there.

In [1]:
from pathlib import Path
import importlib.metadata
import os
import platform
import time

import numpy as np
import pandas as pd
import matplotlib
from matplotlib.colors import to_hex
from threadpoolctl import threadpool_limits
from IPython.display import display
from study_io import ROOT, load_snapshot, new_run_id, begin_calculation, publish_calculation

VERSIONS = {'python': platform.python_version(), **{
    name: importlib.metadata.version(name)
    for name in ('numpy', 'scipy', 'h5py', 'matplotlib', 'pandas', 'threadpoolctl')
}}
print('Runtime versions:', VERSIONS)
from parallel_calculation import simulate_parallel

Runtime versions: {'python': '3.12.3', 'numpy': '1.26.4', 'scipy': '1.17.1', 'h5py': '3.16.0', 'matplotlib': '3.10.9', 'pandas': '3.0.5', 'threadpoolctl': '3.6.0'}


## Exact field snapshot

The snapshot contains the original normalized mean and first positive-frequency
mode, before gyroaveraging. Construction: $B=1.5$ T, characteristic length
0.06 m, source selection `(0, 1)`, cubic interpolation, no denoising or resampling.
Its checksum must match before any numerical code is imported.

In [2]:
potential, FIELD_PROVENANCE, SNAPSHOT_SHA256 = load_snapshot()
from dynamics import GuidingCenterDynamics
from initial_conditions import GCInitialConfiguration
from simulation import BM4Implicit, InitialValueProblem, SimulationRequest, simulate
print('Verified snapshot:', SNAPSHOT_SHA256)
print('Field shape:', potential.grid.shape, '| frequencies:', potential.frequencies)

Verified snapshot: eae45bb3f0c1ff0ee7d27985eaee6fe753dfc1cc2353e1ae75d3a66e09ed4e0d
Field shape: (256, 256) | frequencies: [1.]


## Definition of the section and editable parameters

The effective Hamiltonian is the gyroaveraged potential
$H(x,y,t)=\langle\Phi\rangle_\rho(x,y,t)$, with project convention

$$\dot{x}=-\partial_y H,\qquad \dot{y}=\partial_x H.$$

Positions form the two-dimensional phase plane; velocities follow from this
field and are not independent initial data. The particles do not interact.
Canonical coordinates may be chosen as $(q,p)=(y,x)$.

The retained temporal frequency is exactly one in normalized units:
$H(x,y,t+1)=H(x,y,t)$. The section is the **stroboscopic return map** at forcing
phase zero, $P^k(z_0)=z(kT)$, where $T=1$ and $k=1,\ldots,5000$.
It samples the forcing period, not an individually estimated orbital period
or a crossing of a spatial line. In the extended autonomous formulation this
is the section $t\bmod T=0$ of $K=H+p_t$.

Initial radii are $r_i/L=\mathrm{linspace}(0,0.49,48)$ from the cell centre,
at angle zero towards $+x$. Thus the first particle is exactly at the centre,
and the last is near the periodic edge. The gyro-radius `RHO` is a different
quantity from the initial spatial radius.

Units: $\hat x=2\pi(R-R_0)/\lambda$, $\hat y=2\pi(Z-Z_0)/\lambda$,
$\hat t=t_{\rm SI}/T_0$. The box has normalized length $L\simeq6\pi$.
The return time is one period $T_0$ of the retained mode, read from the field
metadata. Integration uses float64 arithmetic and a constant step
$h=T/20=0.05$; there are 100,000 complete BM4 steps. Internal composition stages
do not count as separate steps.

In [3]:
# Scientific parameters: edit this cell and run all subsequent cells.
N_PARTICLES = 48
N_PROCESSES = 16
N_CYCLES = 5000
STEPS_PER_CYCLE = 20
RADIAL_FRACTIONS = np.linspace(0.0, 0.49, N_PARTICLES)
RADIAL_ANGLE = 0.0             # radians from +x
RHO = 0.3                    # normalized gyro-radius
COUPLING_FREQUENCY = np.pi / 8.0  # BM4 auxiliary-copy coupling
NEWTON_ATOL = 1e-12
NEWTON_RTOL = 1e-11
NEWTON_MAX_ITERATIONS = 40
JACOBIAN_RELATIVE_STEP = float(np.cbrt(np.finfo(np.float64).eps))
CYCLE_DURATION = 1.0
T0 = 0.0
RECORD_NEWTON_HISTORY = True
PROGRESS_EVERY_STEPS = 500
CHECKPOINT_STEPS = 500  # Every 25 cycles; independent of output sampling.
RUN_ID = os.environ.get('POINCARE_RUN_ID') or new_run_id()
VALIDATION_RUN = False  # Set by the CLI only for a disposable smoke test.

In [4]:
for name, value in [('N_PARTICLES', N_PARTICLES), ('N_CYCLES', N_CYCLES),
                    ('STEPS_PER_CYCLE', STEPS_PER_CYCLE), ('N_PROCESSES', N_PROCESSES)]:
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < 1:
        raise ValueError(name + ' must be a positive integer.')
assert N_PROCESSES == 16 and N_PARTICLES >= N_PROCESSES
radii = np.asarray(RADIAL_FRACTIONS, dtype=np.float64)
assert radii.shape == (N_PARTICLES,) and np.all(np.isfinite(radii))
assert np.all((radii >= 0.0) & (radii < 0.5)) and np.all(np.diff(radii) > 0.0)
assert np.isfinite(RADIAL_ANGLE) and T0 == 0.0
assert potential.frequencies.shape == (1,)
np.testing.assert_allclose(potential.frequencies * CYCLE_DURATION, [1.0], rtol=0, atol=1e-14)

L = potential.grid.period
CENTER = np.array([potential.grid.xmin, potential.grid.ymin]) + L / 2
direction = np.array([np.cos(RADIAL_ANGLE), np.sin(RADIAL_ANGLE)])
initial_xy = CENTER + radii[:, None] * L * direction
initial = GCInitialConfiguration.from_components(x=initial_xy[:, 0], y=initial_xy[:, 1])
dynamics = GuidingCenterDynamics(potential, rho=RHO)
problem = InitialValueProblem(dynamics, initial)

# Verify the return phase using both the field and its guiding-centre velocity.
for phase in (0.0, 0.173, 0.637):
    np.testing.assert_allclose(
        dynamics.vector_field(phase, problem.initial_state),
        dynamics.vector_field(phase + CYCLE_DURATION, problem.initial_state),
        rtol=1e-12, atol=1e-12,
    )

H = CYCLE_DURATION / STEPS_PER_CYCLE
N_STEPS = N_CYCLES * STEPS_PER_CYCLE
TF = T0 + N_CYCLES * CYCLE_DURATION
WORKER_SETTINGS = {
    'rho': RHO, 'coupling_frequency': COUPLING_FREQUENCY,
    'newton_atol': NEWTON_ATOL, 'newton_rtol': NEWTON_RTOL,
    'newton_max_iterations': NEWTON_MAX_ITERATIONS,
    'jacobian_relative_step': JACOBIAN_RELATIVE_STEP,
    't_span': [T0, TF], 'step': H, 'n_steps': N_STEPS,
    'record_newton_history': RECORD_NEWTON_HISTORY, 'progress_every': PROGRESS_EVERY_STEPS,
    'checkpoint_steps': CHECKPOINT_STEPS,
    'checkpoint_directory': str(ROOT / 'resultados' / RUN_ID / 'checkpoints'),
    'test_stop_after_chunks': int(os.environ.get('POINCARE_TEST_STOP_AFTER_CHUNKS', '0')) if VALIDATION_RUN else 0,
}

# This identity-to-colour lookup is reused without reordering in every output.
PARTICLE_IDS = np.arange(1, N_PARTICLES + 1)
COLORS = matplotlib.colormaps['turbo'](np.linspace(0.025, 0.975, N_PARTICLES))
COLOR_HEX = [to_hex(color) for color in COLORS]
assert len(set(COLOR_HEX)) == N_PARTICLES
TIME_SCALE_S = FIELD_PROVENANCE['characteristic_period_s']
LENGTH_SCALE_M = FIELD_PROVENANCE['characteristic_length_m'] / (2 * np.pi)

initial_table = pd.DataFrame({
    'particle': PARTICLE_IDS, 'radius_over_L': radii,
    'x0': initial_xy[:, 0], 'y0': initial_xy[:, 1], 'color': COLOR_HEX,
})
print(f'{N_PARTICLES} particles | {N_CYCLES} cycles | {STEPS_PER_CYCLE} steps/cycle')
print(f'h = {H:g}; {N_STEPS} steps; {N_STEPS + 1} stored states per particle')
print(f'Cycle duration = {TIME_SCALE_S:.12g} s; horizon = {TF * TIME_SCALE_S:.12g} s')
print('Float64 epsilon:', np.finfo(np.float64).eps)

display(initial_table)
OUTPUT = begin_calculation(RUN_ID, {
    'method': 'BM4Implicit', 'particle_count': N_PARTICLES,
    'checkpoint_steps': CHECKPOINT_STEPS,
    'process_count': N_PROCESSES, 'partition': 'round-robin by particle ID',
    'blas_threads_per_process': 1, 'start_method': 'spawn',
    'cycles': N_CYCLES, 'steps_per_cycle': STEPS_PER_CYCLE,
    'radial_fractions': radii.tolist(), 'radial_angle_rad': RADIAL_ANGLE,
    'rho': RHO, 'coupling_frequency': COUPLING_FREQUENCY,
    'newton_atol': NEWTON_ATOL, 'newton_rtol': NEWTON_RTOL,
    'newton_max_iterations': NEWTON_MAX_ITERATIONS,
    'jacobian_method': 'analytic', 'jacobian_relative_step': JACOBIAN_RELATIVE_STEP,
    't_span': [T0, TF], 'step': H, 'cycle_duration': CYCLE_DURATION,
    'snapshot_sha256': SNAPSHOT_SHA256, 'versions': VERSIONS,
    'validation_run': VALIDATION_RUN, 'reference_computed': False,
})
print('Run ID:', RUN_ID)
print('Numerical output directory:', OUTPUT)

48 particles | 5000 cycles | 20 steps/cycle
h = 0.05; 100000 steps; 100001 stored states per particle
Cycle duration = 0.00261352184253 s; horizon = 13.0676092127 s
Float64 epsilon: 2.220446049250313e-16


,particle,radius_over_L,x0,y0,color
0,1,0.000000,9.424778,9.424778,#372466
1,2,0.010426,9.621295,9.424778,#3c3286
2,3,0.020851,9.817811,9.424778,#4040a2
3,4,0.031277,10.014328,9.424778,#434eba
4,5,0.041702,10.210845,9.424778,#455ed3
5,6,0.052128,10.407361,9.424778,#466be3
6,7,0.062553,10.603878,9.424778,#4778f0
7,8,0.072979,10.800394,9.424778,#4685fa
8,9,0.083404,10.996911,9.424778,#4391fe
9,10,0.093830,11.193428,9.424778,#3d9efe


Run ID: aws_48p_5000c_20s_16proc_spot_20260920
Numerical output directory: /home/ubuntu/poincare/Poincare_BM4_48_radiales_5000_ciclos_20_steps_16_procesos_spot/resultados/aws_48p_5000c_20s_16proc_spot_20260920


## Integrate sixteen independent groups with the original BM4Implicit

The 48 non-interacting particles are split by round-robin ID into sixteen groups
of three. Sixteen spawned OS processes each advance their complete group
through the same 100,000 steps. Each process uses one BLAS/OpenMP thread and its
own potential/interpolator. A start barrier ensures all sixteen workers are ready
before integration; distinct PIDs and overlapping integration intervals are
asserted and recorded. Results are merged into original particle order.

The project's BM4 implementation is unchanged: twelve explicit stages in the
doubled internal space and one implicit reduced Hairer projection per complete
step, using analytic guiding-centre Jacobians. Its stopping rule is
`NEWTON_ATOL + NEWTON_RTOL * max(1, ||group_state_before||_inf)`.
The norm is evaluated separately for each group; consequently Newton stopping
and roundoff may differ from the earlier joint 48-particle calculation.
Diagnostics are retained with shape `(worker, step)`, never combined into a
misleading single residual/tolerance pair.

All trajectory nodes are saved. Integer stride `STEPS_PER_CYCLE` selects actual
endpoints for the section. Float64 arithmetic, Newton convergence and global
trajectory error are distinct. No reference or refinement study is computed.


## Checkpoints and Spot resumption
Each worker owns three newly spaced radial particles. Every 500 complete steps (25 cycles), it atomically commits a compressed checkpoint with all states, Newton iterates and multiplier norms. Checkpoint fingerprints bind the field, initial conditions, tolerances, step, horizon and driver source.
The restart driver calls the frozen BM4 step at the original global time `T0 + k*H`. It does not reset the forcing phase or interpolate a restart state. Re-run the same ID with `python run.py calculate --resume --run-id <ID>`. Completed numerical files are verified and never reintegrated. In AWS the committed chunks are copied to private S3 before the manifest, so an interrupted upload cannot advertise an incomplete checkpoint.
Progress and timing records describe each worker; a resumed attempt records both reused and newly computed steps. No reference solution is computed, following the explicit request.


## Newton convergence and projection multiplier records
An observational callback reads the residual infinity norm and multiplier infinity norm at every Newton iterate, including iteration zero. It does not repeat BM4 field/map evaluations or change the accepted iterate. The original snapshot remains unchanged.
`newton_steps.csv.gz` stores one row per worker and complete step: correction count, residual evaluations, initial/final residual, tolerance, residual/tolerance and final $\|\mu\|_\infty$. `newton_iterations.csv.gz` stores the complete iteration history. Both CSV files and their NPZ counterparts are checksummed and included in the AWS result archive.
The multiplier norm belongs to a particle group, not an individual particle. Newton convergence is distinct from integration accuracy. As requested earlier, no reference solution is computed. The step is 2.5 times larger than in the previous 50-step-per-cycle study.


In [5]:
solution = simulate_parallel(initial_xy, WORKER_SETTINGS, processes=N_PROCESSES)
RUNTIME_SECONDS = solution.parallel_wall_seconds
print(f'Parallel BM4 completed in {RUNTIME_SECONDS:.2f} s, including worker startup and merge.')
print(f'All sixteen workers integrated simultaneously for {solution.simultaneous_integration_seconds:.2f} s.')
display(pd.DataFrame(solution.workers)[['worker', 'pid', 'particle_ids',
                                     'integration_seconds', 'cpu_seconds']])


Worker 9: resumed verified step 75000/100000
Worker 2: resumed verified step 76500/100000
Worker 3: resumed verified step 77500/100000
Worker 15: resumed verified step 76000/100000
Worker 8: resumed verified step 76500/100000
Worker 16: resumed verified step 80500/100000
Worker 4: resumed verified step 80000/100000
Worker 7: resumed verified step 83000/100000
Worker 5: resumed verified step 75000/100000
Worker 13: resumed verified step 78000/100000
Worker 10: resumed verified step 86500/100000
Worker 6: resumed verified step 90500/100000
Worker 1: resumed verified step 75500/100000
Worker 11: resumed verified step 89500/100000
Worker 12: resumed verified step 94000/100000
Worker 14: resumed verified step 96000/100000


Worker 14: checkpoint 96500/100000 (96.5%), t=4825


Worker 11: checkpoint 90000/100000 (90.0%), t=4500


Worker 6: checkpoint 91000/100000 (91.0%), t=4550


Worker 12: checkpoint 94500/100000 (94.5%), t=4725


...

Worker 5: checkpoint 99500/100000 (99.5%), t=4975


Worker 9: checkpoint 100000/100000 (100.0%), t=5000
Worker 9/16: PID 6423, 3 particles, 2613.58 s


Worker 5: checkpoint 100000/100000 (100.0%), t=5000
Worker 5/16: PID 6422, 3 particles, 2624.19 s


Parallel BM4 completed in 2624.96 s, including worker startup and merge.
All sixteen workers integrated simultaneously for 259.84 s.


,worker,pid,particle_ids,integration_seconds,cpu_seconds
0,1,6417,"[1, 17, 33]",2498.130011,2497.763755
1,2,6419,"[2, 18, 34]",2374.077541,2373.733158
2,3,6413,"[3, 19, 35]",2331.087731,2330.711770
3,4,6420,"[4, 20, 36]",2038.772080,2038.294829
4,5,6422,"[5, 21, 37]",2624.187926,2623.841605
5,6,6418,"[6, 22, 38]",711.777423,711.641988
6,7,6416,"[7, 23, 39]",1752.760992,1752.360765
7,8,6427,"[8, 24, 40]",2235.828543,2235.146926
8,9,6423,"[9, 25, 41]",2613.580223,2612.952216
9,10,6421,"[10, 26, 42]",1401.246355,1401.114829


In [6]:
times = solution.t
# Shapes: states=(2*N, steps+1), xy=(steps+1, N, 2).
states = solution.states
xy = np.stack(solution.positions(), axis=-1).transpose(1, 0, 2)
assert xy.shape == (N_STEPS + 1, N_PARTICLES, 2)
assert states.dtype == np.float64 and np.all(np.isfinite(xy))
assert solution.diagnostics['step_count'] == N_STEPS
assert solution.diagnostics['projection_solver_formulation'] == 'bm4_implicit_reduced'
assert solution.diagnostics['newton_jacobian_method'] == 'analytic'
np.testing.assert_allclose(np.diff(times), H, rtol=0, atol=64*np.finfo(float).eps*max(1, TF))
np.testing.assert_array_equal(xy[0], initial_xy)

cycle_nodes = np.arange(N_CYCLES + 1) * STEPS_PER_CYCLE
cycle_times = times[cycle_nodes]
np.testing.assert_allclose(cycle_times, T0 + np.arange(N_CYCLES + 1)*CYCLE_DURATION,
                           rtol=0, atol=64*np.finfo(float).eps*max(1, TF))
cycle_xy = xy[cycle_nodes]  # Includes initial data only at index zero.
domain_origin = np.array([potential.grid.xmin, potential.grid.ymin])
cycle_xy_wrapped = (cycle_xy - domain_origin) % L + domain_origin
section_xy = cycle_xy_wrapped[1:]
assert section_xy.shape == (N_CYCLES, N_PARTICLES, 2)

residuals = np.asarray(solution.diagnostics['nonlinear_residual_norms'])
tolerances = np.asarray(solution.diagnostics['nonlinear_tolerances'])
iterations = np.asarray(solution.diagnostics['nonlinear_iterations'])
assert residuals.shape == tolerances.shape == iterations.shape == (N_PROCESSES, N_STEPS)
assert np.all(np.isfinite(residuals)) and np.all(tolerances > 0)
assert np.all(residuals <= tolerances * (1 + 32*np.finfo(float).eps))
print(f'Validated: {N_CYCLES * N_PARTICLES} section points, {N_CYCLES} returns per particle.')
print(f'Maximum residual / tolerance: {np.max(residuals/tolerances):.6g}')
print(f'Newton corrections per group and step: mean {iterations.mean():.3f}, max {iterations.max()}')

Validated: 240000 section points, 5000 returns per particle.
Maximum residual / tolerance: 0.999996
Newton corrections per group and step: mean 1.796, max 2


## Assemble positions and publish numerical outputs

The table contains one row for each particle and completed cycle. Cycle zero
is exported separately. The NPZ contains every step, cycle samples, colours and
Newton diagnostics. `COMPLETE.json` is published last, with checksums of all
numerical files; visualization refuses incomplete or altered results.

Only numerical outputs and text are generated here. `visualizacion.ipynb` reads
these saved files independently, using the stored parameters and colour mapping.

In [7]:
all_cycle_positions = pd.DataFrame({
    'cycle': np.repeat(np.arange(N_CYCLES + 1), N_PARTICLES),
    'time_normalized': np.repeat(cycle_times, N_PARTICLES),
    'time_s': np.repeat(cycle_times * TIME_SCALE_S, N_PARTICLES),
    'particle': np.tile(PARTICLE_IDS, N_CYCLES + 1),
    'color': np.tile(COLOR_HEX, N_CYCLES + 1),
    'initial_radius_over_L': np.tile(radii, N_CYCLES + 1),
    'x_unwrapped': cycle_xy[..., 0].ravel(),
    'y_unwrapped': cycle_xy[..., 1].ravel(),
    'x_wrapped': cycle_xy_wrapped[..., 0].ravel(),
    'y_wrapped': cycle_xy_wrapped[..., 1].ravel(),
    'x_over_L': cycle_xy_wrapped[..., 0].ravel() / L,
    'y_over_L': cycle_xy_wrapped[..., 1].ravel() / L,
})
all_cycle_positions['R_wrapped_m'] = (FIELD_PROVENANCE['source_origin_m'][0]
                                     + all_cycle_positions['x_wrapped'] * LENGTH_SCALE_M)
all_cycle_positions['Z_wrapped_m'] = (FIELD_PROVENANCE['source_origin_m'][1]
                                     + all_cycle_positions['y_wrapped'] * LENGTH_SCALE_M)
cycle_positions = all_cycle_positions.loc[all_cycle_positions['cycle'] > 0].copy()
assert len(cycle_positions) == N_CYCLES * N_PARTICLES
assert np.all(cycle_positions.groupby('particle').size().to_numpy() == N_CYCLES)
assert np.all(all_cycle_positions.groupby('particle')['color'].nunique().to_numpy() == 1)

def positions_at_cycle(cycle):
    """Return one complete particle table at a validated integer return index."""
    if isinstance(cycle, (bool, np.bool_)) or not isinstance(cycle, (int, np.integer)):
        raise ValueError('Cycle must be an integer.')
    if not 0 <= cycle <= N_CYCLES:
        raise ValueError(f'Cycle must be between 0 and {N_CYCLES}.')
    return all_cycle_positions.loc[all_cycle_positions['cycle'] == cycle].reset_index(drop=True)

display(positions_at_cycle(N_CYCLES))

,cycle,time_normalized,time_s,particle,color,initial_radius_over_L,x_unwrapped,y_unwrapped,x_wrapped,y_wrapped,x_over_L,y_over_L,R_wrapped_m,Z_wrapped_m
0,5000,5000.0,13.067609,1,#372466,0.000000,10.112494,9.507355,10.112494,9.507355,0.536484,0.504381,0.097270,0.091140
1,5000,5000.0,13.067609,2,#3c3286,0.010426,9.744853,9.605378,9.744853,9.605378,0.516980,0.509581,0.093760,0.092076
2,5000,5000.0,13.067609,3,#4040a2,0.020851,9.566434,9.469888,9.566434,9.469888,0.507515,0.502393,0.092056,0.090782
3,5000,5000.0,13.067609,4,#434eba,0.031277,10.053251,9.471687,10.053251,9.471687,0.533342,0.502489,0.096705,0.090800
4,5000,5000.0,13.067609,5,#455ed3,0.041702,10.201898,9.384446,10.201898,9.384446,0.541228,0.497860,0.098124,0.089966
5,5000,5000.0,13.067609,6,#466be3,0.052128,9.313315,8.838960,9.313315,8.838960,0.494087,0.468921,0.089639,0.084757
6,5000,5000.0,13.067609,7,#4778f0,0.062553,8.739132,7.266116,8.739132,7.266116,0.463625,0.385479,0.084156,0.069738
7,5000,5000.0,13.067609,8,#4685fa,0.072979,8.764410,5.407554,8.764410,5.407554,0.464966,0.286880,0.084397,0.051990
8,5000,5000.0,13.067609,9,#4391fe,0.083404,12.706070,13.655436,12.706070,13.655436,0.674078,0.724443,0.122037,0.130751
9,5000,5000.0,13.067609,10,#3d9efe,0.093830,6.153423,11.633344,6.153423,11.633344,0.326449,0.617168,0.059464,0.111442


In [8]:
run_metadata = {
    'method': 'BM4Implicit', 'nonlinear_solver': 'newton',
    'projection': 'one reduced Hairer projection per complete BM4 step',
    'newton_jacobian': 'analytic', 'reference_computed': False,
    'trajectory_accuracy_certified': False,
    'arithmetic': 'float64', 'machine_epsilon': float(np.finfo(float).eps),
    'particle_count': N_PARTICLES, 'cycles': N_CYCLES,
    'steps_per_cycle': STEPS_PER_CYCLE, 'complete_steps': N_STEPS,
    't_span': [T0, TF], 'cycle_duration': CYCLE_DURATION, 'step': H,
    'section_phase': 0.0, 'section_excludes_initial_state': True,
    'radial_fractions': radii.tolist(), 'radial_angle_rad': RADIAL_ANGLE,
    'rho': RHO, 'coupling_frequency': COUPLING_FREQUENCY,
    'newton_atol': NEWTON_ATOL, 'newton_rtol': NEWTON_RTOL,
    'newton_max_iterations': NEWTON_MAX_ITERATIONS,
    'jacobian_relative_step': JACOBIAN_RELATIVE_STEP,
    'residual_tolerance_formula': 'atol + rtol * max(1, infinity_norm(group_state_before))',
    'maximum_residual_to_tolerance': float(np.max(residuals/tolerances)),
    'mean_newton_corrections': float(iterations.mean()),
    'maximum_newton_corrections': int(iterations.max()),
    'runtime_seconds': RUNTIME_SECONDS,
    'runtime_scope': 'current attempt: checkpoint loading, new steps, checkpoint writing, pool and merge',
    'process_count': N_PROCESSES, 'start_method': 'spawn',
    'blas_threads_per_process': 1, 'partition': 'round-robin by particle ID',
    'workers': solution.workers,
    'simultaneous_integration_seconds': solution.simultaneous_integration_seconds,
    'diagnostics_layout': '[worker, complete step within each group]',
    'nonlinear_stopping_scope': 'separate group state infinity norm',
    'host_cpu_count': os.cpu_count(),
    'host_cpu_affinity': sorted(os.sched_getaffinity(0)),
    'time_scale_s': TIME_SCALE_S, 'length_scale_m': LENGTH_SCALE_M,
    'state_layout': '[x_1,...,x_N,y_1,...,y_N] by saved time',
    'cycle_positions_layout': '[cycle including zero, particle, coordinate x/y]',
    'colours': dict(zip(map(str, PARTICLE_IDS), COLOR_HEX)),
    'versions': VERSIONS, 'snapshot_sha256': SNAPSHOT_SHA256,
    'field_provenance': FIELD_PROVENANCE,
}

run_metadata.update(schema_version=1, run_id=RUN_ID, validation_run=VALIDATION_RUN)
arrays = {
    'times': times, 'states': states,
    'cycle_times': cycle_times, 'cycle_positions': cycle_xy,
    'cycle_positions_wrapped': cycle_xy_wrapped,
    'initial_positions': initial_xy, 'particle_ids': PARTICLE_IDS,
    'colors_rgba': COLORS, 'colors_hex': np.asarray(COLOR_HEX),
    'nonlinear_iterations': iterations, 'nonlinear_residuals': residuals,
    'nonlinear_tolerances': tolerances,
    'projection_multiplier_norms': solution.diagnostics['projection_multiplier_norms'],
    **{key: solution.diagnostics[key] for key in (
        'newton_history_offsets', 'newton_history_residuals', 'newton_history_mu_norms')},
}
run_metadata['checkpoint_steps'] = CHECKPOINT_STEPS
run_metadata['checkpoint_driver_sha256'] = __import__('study_io').digest(ROOT / 'checkpoint_calculation.py')
run_metadata['initial_grid_is_new'] = True
run_metadata['newton_history_recorded'] = True
run_metadata['newton_history_layout'] = 'Absolute offsets [worker, step boundary] into flat iteration histories'
run_metadata['newton_iteration_zero'] = 'Initial zero multiplier, before any correction'
run_metadata['mu_norm_definition'] = 'Infinity norm of the reduced projection multiplier of each particle group'
run_metadata['diagnostic_source'] = 'Observational callback in a private copy of the verified frozen Newton solver'
run_metadata['diagnostic_instrumentation_sha256'] = __import__('study_io').digest(ROOT / 'newton_diagnostics.py')
run_metadata['newton_mu_summary'] = []
for w in range(N_PROCESSES):
    mu = arrays['projection_multiplier_norms'][w]
    run_metadata['newton_mu_summary'].append({
        'worker': w + 1, 'mean_newton_corrections': float(iterations[w].mean()),
        'max_newton_corrections': int(iterations[w].max()),
        'total_newton_corrections': int(iterations[w].sum()),
        'total_residual_evaluations': int((iterations[w] + 1).sum()),
        'max_residual_over_tolerance': float((residuals[w] / tolerances[w]).max()),
        'mu_inf_mean': float(mu.mean()), 'mu_inf_rms': float(np.sqrt(np.mean(mu ** 2))),
        'mu_inf_max': float(mu.max()), 'mu_inf_final': float(mu[-1]),
    })
publish_calculation(OUTPUT, run_metadata, arrays, cycle_positions, positions_at_cycle(0))
print(f'Complete: {N_PARTICLES} particles, {N_CYCLES} cycles, {N_STEPS} steps.')
print(f'Saved {len(cycle_positions)} return positions to {OUTPUT}')
print('Next: open visualizacion.ipynb and select this run ID.')

Complete: 48 particles, 5000 cycles, 100000 steps.
Saved 240000 return positions to /home/ubuntu/poincare/Poincare_BM4_48_radiales_5000_ciclos_20_steps_16_procesos_spot/resultados/aws_48p_5000c_20s_16proc_spot_20260920
Next: open visualizacion.ipynb and select this run ID.
